# Context & State

In a real app, there are two types of information involved:

- Information your Python server knows before the chat starts (e.g., user_id, account_balance, auth_token). The user didn't type this in; your backend database already has it.

- Information created during the chat conversation (e.g., the message history, or the status of a money transfer the bot is performing).

This difference is the entire reason Context and State exist.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## State

This is the live memory built through out the LLM and user conversation

State is read/write both. Means it gets updated when a particular information is revealed by the user.

For e.g: `messages`, `favorite color`, `verification step`, `cart items`

Back to our Banking AI Support Chatbot example

Lets say the user says _"Transfer $500 to Bob."_ and our bot guides through the transfer and needs to keep track of the `transfer_status` variable which is: `None`, `PENDING CONFIRMATION`, `COMPLETED`

Once the bot gets the user message, it changes the variable to `PENDING CONFIRMATION`

Now once the AI prompts user _"Are you sure you want to transfer $500 to Bob?"_ and the user agrees, it changes the variable status to `COMPLETED`

This is called State. As it is changing/updating as per the LLM/user conversation

In [ ]:
# instantiating the memory object in the beginning so it works globally
# and doesn't start fresh when we make the agent again to pass the new tools we made

from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

In [3]:
# by default, the agent keeps track of the ""messages" state 
# but if we want our custom state, we can create a custom state class

from langchain.agents import AgentState
from typing import Literal

class BankAgentState(AgentState):
    transfer_amount : float
    transfer_status : Literal["None", "PENDING CONFIRMATION", "COMPLETED"]

## Write to state

In [ ]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

# here we are initiating the transfer 
# so the transfer status should always be "PENDING CONFIRMATION" unless we call the confirm_transfer tool
# the reason we are passing the transfer_amount variable is that:
# when user says "Send 500 to bob", the agent calls this initiate_transfer tool to updates the state to PENDING CONFIRMATION
# and prompts the user to confirm. now in the next turn the user doesn't again mention 500, they either say "YES"
# now as the messages state is being made, the agent knows that 500 is to be transferred. the parameter that we took
@tool
def inititate_transfer(transfer_amount: float, runtime: ToolRuntime) -> Command:

    """initiate transfer of the money and update the transfer status to pending"""

    return Command(update={
        "transfer_amount": transfer_amount,
        "transfer_status": "PENDING CONFIRMATION", 
        "messages": [ToolMessage(f"{transfer_amount} is on pending to send. Need Confirmation!", tool_call_id=runtime.tool_call_id)]}
        )

# this just updates the transfer_status to COMPLETED
@tool
def confirm_transfer(transfer_amount: float, runtime: ToolRuntime) -> Command:
    """Confirm user about the money transfer and set transfer_status to COMPLETED"""

    return Command(update={
        "transfer_status":"COMPLETED",
        "messages": [ToolMessage(f"Amount of Rs.{transfer_amount} Transferred!", tool_call_id=runtime.tool_call_id)]
    })


In [5]:
from langchain.agents import create_agent

system_prompt = """
ANSWER VERY BRIEFLY. NO LENGTHY EXPLANATIONS.

Help users with their Banking operations.

UNDERSTAND THE QUERY THEN MUST USE YOUR TOOLS WHERE NECESSARY

When User wants to send money to someone:
1. First make sure you know the correct amount to send
2. Initialise the transfer and ask for confirmation
3. Once they agree and confirm, only then send the money and tell user that the money is sent
4. If they don't confirm or agree, do not send the money.

DO NOT SEND THE MONEY ON FIRST ASK. 
Always ask the user for confirmation before sending money

Use your tools wisely to initialize transfer and confirm transfer and to read the status of transfer
"""

agent = create_agent(
    "gpt-5-nano",
    tools=[inititate_transfer, confirm_transfer],
    system_prompt=system_prompt,
    checkpointer=memory,
    state_schema=BankAgentState
)

In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    { "messages": [HumanMessage(content="Send rs.700 to Bob")]},
    {"configurable": {"thread_id": "321"}}
)

In [7]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Send rs.700 to Bob', additional_kwargs={}, response_metadata={}, id='869fafcb-ad79-469b-93f9-fbdc1650124c'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 474, 'prompt_tokens': 318, 'total_tokens': 792, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E9spr4fdjkPEWjMpVqGW2LNq6QEWk', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fd75a-46a4-7a40-810a-409fc8899223-0', tool_calls=[{'name': 'inititate_transfer', 'args': {'transfer_amount': 700}, 'id': 'call_h0UZrXJ70XWMZyypM2u6Oh1b', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 318

In [8]:
pprint(response["messages"][-1].content)

(' Rs 700 to Bob is pending. Please confirm to complete the transfer.\n'
 ' Reply with: CONFIRM')


In [9]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="yes I confirm. pls send")]
    },
    {"configurable": {"thread_id": "321"}}
)

pprint(response)

{'messages': [HumanMessage(content='Send rs.700 to Bob', additional_kwargs={}, response_metadata={}, id='869fafcb-ad79-469b-93f9-fbdc1650124c'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 474, 'prompt_tokens': 318, 'total_tokens': 792, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E9spr4fdjkPEWjMpVqGW2LNq6QEWk', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fd75a-46a4-7a40-810a-409fc8899223-0', tool_calls=[{'name': 'inititate_transfer', 'args': {'transfer_amount': 700}, 'id': 'call_h0UZrXJ70XWMZyypM2u6Oh1b', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 318

In [10]:
pprint(response["messages"][-1].content)

('Transfer of Rs 700 to Bob completed. Would you like to perform another '
 'transfer?')


## Read state

As LLMs has a certain size of their context window.

This read state tool will help the agent to read the state even if the messages are gone out of the context window

In [11]:
@tool
def read_transfer_status(runtime: ToolRuntime) -> str:
    """Read the transfer status of the mony sent, from the state."""
    try:
        return runtime.state["transfer_status"]
    except KeyError:
        return "Can't fetch transfer status"

agent = create_agent(
    "gpt-5-nano",
    tools=[inititate_transfer, confirm_transfer, read_transfer_status],
    checkpointer=memory,
    state_schema=BankAgentState
)

In [12]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's the transfer status of the money I sent to bob?")]},
    {"configurable": {"thread_id": "321"}}
)

pprint(response)

{'messages': [HumanMessage(content='Send rs.700 to Bob', additional_kwargs={}, response_metadata={}, id='869fafcb-ad79-469b-93f9-fbdc1650124c'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 474, 'prompt_tokens': 318, 'total_tokens': 792, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-E9spr4fdjkPEWjMpVqGW2LNq6QEWk', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fd75a-46a4-7a40-810a-409fc8899223-0', tool_calls=[{'name': 'inititate_transfer', 'args': {'transfer_amount': 700}, 'id': 'call_h0UZrXJ70XWMZyypM2u6Oh1b', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 318

In [14]:
pprint(response["messages"][-1].content)

('The transfer of Rs 700 to Bob is COMPLETED. Would you like a receipt or to '
 'make another transfer?')
